# US Flight Delays and Cancellations: Prediction Report_2

Professor: Yunlong Wang
Author: Stephen Handy
Course: CSC381-01
Data: Fall 2026

## Abstract

This report investigates whether 2025-2026 flight data can be used to predict flight delays or cancellations before departure. Analysis includes data, preprocessing decisions, models, evaluation results, and limitations.

## Research Question

Can a flight's delay or cancellation status be accurately predicted using information available pre-flight departure?

## Hypothesis

Time, route, airline, airport, and scheduled-flight characteristics will provide enough information to predict cancellations and delays, although unforeseen variables — such as weather and operational variables — will limit the accuracy of these predictions. 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path("dataset/flights_sample.csv")

df = pd.read_csv(csv_path)
print(df.shape)
df.head()


(500000, 40)


,flight_date,year,month,day_of_week,airline_code,airline,tail_number,flight_number,origin,origin_city,...,diverted,sched_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2025-01-01,2025,1,3,AA,American Airlines,N960NN,2158,SMF,"Sacramento, CA",...,0,249,221.0,202.0,1781,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,2025,1,3,B6,JetBlue Airways,N3145J,98,DEN,"Denver, CO",...,0,226,215.0,180.0,1626,0.0,0.0,0.0,0.0,96.0
2,2025-01-01,2025,1,3,AS,Alaska Airlines,N973AK,84,ANC,"Anchorage, AK",...,0,213,192.0,176.0,1448,NaN,NaN,NaN,NaN,NaN
3,2025-01-01,2025,1,3,AS,Alaska Airlines,N296AK,704,SEA,"Seattle, WA",...,0,152,128.0,110.0,867,NaN,NaN,NaN,NaN,NaN
4,2025-01-01,2025,1,3,B6,JetBlue Airways,N955JB,2169,BOS,"Boston, MA",...,0,212,204.0,184.0,1237,NaN,NaN,NaN,NaN,NaN


In [2]:
df = df[
    (df["cancelled"] == 0)
    & (df["diverted"] == 0)
    & (df["arr_delay"].notna())
].copy()


In [3]:
df["sched_dep_hour"] = df["sched_dep_time"] // 100


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

feature_columns = [
    "airline_code", "origin", "dest", "month", "day_of_week", "sched_dep_hour", "dep_time_block", "distance", "sched_elapsed_time"
]

categorical = [
    "airline_code", "origin", "dest", "month", "day_of_week", "dep_time_block"
]

numeric = ["sched_dep_hour", "distance", "sched_elapsed_time"]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", "passthrough", numeric)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

df["flight_date"] = pd.to_datetime(df["flight_date"])
df = df.sort_values("flight_date")

split_date = df["flight_date"].quantile(0.80)

train = df[df["flight_date"] <= split_date].copy()
test = df[df["flight_date"] > split_date].copy()

X_train = train[feature_columns]
y_train = train["arr_delay"]

X_test = test[feature_columns]
y_test = test["arr_delay"]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


Training rows: 392438
Testing rows: 97198
MAE: 31.655815190220064
RMSE: 64.86979731749867
R²: 0.029068151843322787
